<a href="https://colab.research.google.com/github/muhammadhasaan82/ai-property-booking-bias-variance-analysis/blob/main/Bias%26Variace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 22.8 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
pip install pandas openpyxl

In [3]:
pip install --upgrade keras


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 5.7 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.13.2
    Uninstalling keras-3.13.2:
      Successfully uninstalled keras-3.13.2


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**# STEP: 1**
`Performing EDA`

In [5]:
import pandas as pd

In [6]:
import matplotlib.pyplot as plt

In [7]:
file_path = '/content/drive/MyDrive/AI Property Booking Concierge/dataset.csv'
df = pd.read_csv(file_path)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor as RFR
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.svm import SVR
from sklearn.metrics import (accuracy_score,confusion_matrix,ConfusionMatrixDisplay,mean_squared_error,r2_score)
from keras.optimizers import Adam
import time, warnings
import numpy as np

In [9]:
print(df.shape)
df.head(
)

(25000, 20)


,id,title,description,property_type,bedrooms,bathrooms,price_per_night,square_feet,amenities,city,state,zipcode,country,occupancy_max,rating,reviews_count,availability_start,availability_end,host_id,created_at
0,1634f9de-786c-4810-a552-604f378bc84b,5BR Loft in Miami,Spacious layout ideal for groups and longer st...,Loft,5,5,727,2085,balcony;dryer;gym;kitchen;washer,Miami,FL,32520,USA,10,3.64,47,2025-11-23,2025-12-04,h_1434,2024-12-06T15:35:49
1,2a9e8a58-969c-46e4-8ce1-0a14cb3fde3d,3BR House in Somerville,Cozy retreat with fully equipped kitchen and w...,House,3,3,411,1284,dryer;fireplace;parking;pool;wheelchair_access,Somerville,MA,6552,USA,6,3.82,916,2025-11-02,2025-11-05,h_2519,2025-03-09T15:35:49
2,f8b59c5e-ed67-4d69-81d6-e6e35634bc5e,2BR Duplex in Fort Lauderdale,Quiet residential area with quick access to do...,Duplex,2,2,245,722,ac;balcony;elevator;heating;kitchen;pet_friend...,Fort Lauderdale,FL,32750,USA,4,4.53,597,2025-09-30,2025-10-05,h_2654,2025-03-09T15:35:49
3,684d1ee5-369a-4ce4-8223-53344dc3bed1,5BR Studio in Roswell,Charming home with a private garden and outdoo...,Studio,5,5,359,2527,dryer;fireplace;heating;kitchen;washer;wheelch...,Roswell,GA,32803,USA,10,4.35,59,2025-11-18,2025-11-26,h_5422,2024-10-27T15:35:49
4,5f189ec6-199c-4d52-b7c1-f45a699f2687,5BR Studio in Elizabeth,"Walkable location with shops, cafes, and night...",Studio,5,5,522,3389,ac;elevator;pet_friendly;pool;wifi,Elizabeth,NJ,8084,USA,10,3.90,889,2025-10-30,2025-11-04,h_9179,2025-03-01T15:35:49


In [10]:
print(df.columns)

Index(['id', 'title', 'description', 'property_type', 'bedrooms', 'bathrooms',
       'price_per_night', 'square_feet', 'amenities', 'city', 'state',
       'zipcode', 'country', 'occupancy_max', 'rating', 'reviews_count',
       'availability_start', 'availability_end', 'host_id', 'created_at'],
      dtype='object')


In [11]:
print(df.dtypes)

id                     object
title                  object
description            object
property_type          object
bedrooms                int64
bathrooms               int64
price_per_night         int64
square_feet             int64
amenities              object
city                   object
state                  object
zipcode                 int64
country                object
occupancy_max           int64
rating                float64
reviews_count           int64
availability_start     object
availability_end       object
host_id                object
created_at             object
dtype: object


In [12]:
pivot_table = pd.pivot_table(
    df,
    index=["city"],
    columns="property_type",
    values="id",
    aggfunc="count",
    fill_value=0,
    margins=True,
    margins_name="Grand Total"
)

print(pivot_table)

property_type  Apartment  Bungalow  Condo  Cottage  Duplex  House  Loft  \
city                                                                      
Akron                 13        10     19       17      12     17     8   
Albany                10        11      9       13      13     15     9   
Alexandria            15        12      9       22      11     14    13   
Allentown             13        16      7       22      13     12    12   
Alpharetta            13        14     13       12      14     11    16   
...                  ...       ...    ...      ...     ...    ...   ...   
Worcester             11        11     11        6      11     10    14   
Yonkers               13        17      7       14      10     18    18   
York                  13        15     12       16      13     11     7   
Youngstown             9        12     12       12      10     13    14   
Grand Total         2537      2452   2550     2573    2475   2538  2481   

property_type  Studio  T

In [13]:
pivot_table_one = pd.pivot_table(
    df,
    index="city",
    columns="property_type",
    values="id",
    aggfunc="count",
    fill_value=0,
    margins=True,
    margins_name="Grand Total"
)
print(pivot_table_one)

property_type  Apartment  Bungalow  Condo  Cottage  Duplex  House  Loft  \
city                                                                      
Akron                 13        10     19       17      12     17     8   
Albany                10        11      9       13      13     15     9   
Alexandria            15        12      9       22      11     14    13   
Allentown             13        16      7       22      13     12    12   
Alpharetta            13        14     13       12      14     11    16   
...                  ...       ...    ...      ...     ...    ...   ...   
Worcester             11        11     11        6      11     10    14   
Yonkers               13        17      7       14      10     18    18   
York                  13        15     12       16      13     11     7   
Youngstown             9        12     12       12      10     13    14   
Grand Total         2537      2452   2550     2573    2475   2538  2481   

property_type  Studio  T

In [14]:
df["n_amenities"] = df["amenities"].str.split(";").apply(len)
df["availability_start"] = pd.to_datetime(df["availability_start"])
df["availability_end"] = pd.to_datetime(df["availability_end"])
df["stay_days"] = (df["availability_end"] - df["availability_start"]).dt.days

In [15]:
NUMERIC = ["bedrooms", "bathrooms", "square_feet", "occupancy_max", "rating", "reviews_count", "stay_days", "n_amenities"]

CATEGORICAL = ["property_type", "city"]
TARGET = "price_per_night"

# STEP: 2


```
Feature Engineering
```



In [16]:
X = df[NUMERIC + CATEGORICAL]
y = df[TARGET]

In [17]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
])

# Step 3 `Model Builders`

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(X,y,test_size=0.25,random_state=42)

In [19]:
X_test, X_dev, y_test, y_dev = train_test_split(X_temp,y_temp,test_size=0.40,random_state=42)

In [26]:
RNG = 42

# MODEL_BUILDERS = {
#     "LinearRegression": lambda seed=None: LinearRegression(),
#     "Ridge": lambda seed=None: Ridge(alpha=1.0, random_state=seed),
#     "RandomForest": lambda seed=None: RFR(n_estimators=100, max_depth=12, random_state=seed, n_jobs=-1),
#     "SVR(rbf)": lambda seed=None: SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale"),
#     "MLPAdam": lambda seed=None: MLPRegressor(hidden_layer_sizes=(32, 16), solver="adam", max_iter=1000, early_stopping=True, random_state=seed),
#     "MLP-LBFGS": lambda seed=None: MLPRegressor(hidden_layer_sizes=(32, 16), solver="lbfgs", max_iter=1000, random_state=seed),
#     "HistGradientBoosting": lambda seed=None: HistGradientBoostingRegressor(max_iter=100, random_state=seed),
# }

MODEL_BUILDERS = {
    "LinearRegression": lambda seed=None: LinearRegression(),
    "Ridge": lambda seed=None: Ridge(alpha=1.0, random_state=seed),
    "RandomForest": lambda seed=None: RFR(
        n_estimators=100,
        max_depth=12,
        random_state=seed,
        n_jobs=-1
    ),
    "SVR(rbf)": lambda seed=None: SVR(
        kernel="rbf",
        C=10,
        epsilon=0.1,
        gamma="scale"
    ),
    "MLPAdam": lambda seed=None: MLPRegressor(
        hidden_layer_sizes=(32,16),
        solver="adam",
        max_iter=1000,
        early_stopping=True,
        random_state=seed
    ),
    "MLP-LBFGS": lambda seed=None: MLPRegressor(
        hidden_layer_sizes=(32,16),
        solver="lbfgs",
        max_iter=1000,
        random_state=seed
    ),
    "HistGradientBoosting": lambda seed=None: HistGradientBoostingRegressor(
        max_iter=100,
        random_state=seed
    ),
}

In [27]:
#SINGLE TRAIN/TEST SPLIT SCOREBOARD
print("="*70)
print(f"single split scoreboard (train={len(X_train)}, test={len(X_test)}, Dev_set={len(X_dev)})")
print("="*70)

single split scoreboard (train=18750, test=3750, Dev_set=2500)


In [28]:
warnings.filterwarnings("ignore")

scoreboard = []
for name, builder in MODEL_BUILDERS.items():
  pipe = Pipeline([("prep", preprocess), ("model", builder(RNG))])
  t0 = time.time()
  pipe.fit(X_train, y_train)
  pred_tr, pred_dev = pipe.predict(X_train), pipe.predict(X_dev)
  row = {
      "model": name,
      "train_rmse": mean_squared_error(y_train, pred_tr) ** 0.5,
      "dev_rmse": mean_squared_error(y_dev, pred_dev) ** 0.5,
      "train_r2": r2_score(y_train, pred_tr),
      "dev_r2": r2_score(y_dev, pred_dev),
      "fit_time_s": time.time() - t0,
  }

In [29]:
  row["overfit_gap_rmse"] = row ["dev_rmse"] - row["train_rmse"]
  scoreboard.append(row)
  print(f"{name:22s} train_RMSE={row['train_rmse']:7.2f} dev_RMSE={row['dev_rmse']:7.2f}"
        f"train_R2={row['train_r2']:.3f} dev_R2={row['dev_r2']:.3f} ({row['fit_time_s']:.1f}s)")

scoreboard_df = pd.DataFrame(scoreboard)
scoreboard_df

HistGradientBoosting   train_RMSE=  54.53 dev_RMSE=  59.01train_R2=0.916 dev_R2=0.900 (4.5s)


,model,train_rmse,dev_rmse,train_r2,dev_r2,fit_time_s,overfit_gap_rmse
0,HistGradientBoosting,54.527369,59.00903,0.915684,0.899579,4.464917,4.48166


# **bootstrap bias-variance decomposition, evaluated on dev**

In [30]:
BOOTSTRAPS = 5
rng = np.random.RandomState(RNG)
y_dev_arr = np.asarray(y_dev).ravel()


In [32]:
decomp = []

for name, builder in MODEL_BUILDERS.items():
    t0 = time.time()

    preds = np.zeros((BOOTSTRAPS, len(y_dev_arr)))
    n = len(X_train)

    for b in range(BOOTSTRAPS):
        bi = rng.randint(0, n, n)
        Xb, yb = X_train.iloc[bi], y_train.iloc[bi]

        pipe = Pipeline([
            ("prep", preprocess),
            ("model", builder(RNG + b))
        ])

        pipe.fit(Xb, yb)
        preds[b] = pipe.predict(X_dev)

    avg_pred = preds.mean(axis=0)

    bias_sq = float(np.mean((avg_pred - y_dev_arr) ** 2))
    variance = float(np.mean(preds.var(axis=0)))

    decomp.append({
        "model": name,
        "bias_sq_plus_noise": bias_sq,
        "variance": variance,
        "expected_mse": bias_sq + variance,
        "expected_rmse": (bias_sq + variance) ** 0.5,
        "time_s": time.time() - t0,
    })

    print(
        f"{name:22s} "
        f"bias^2(+noise)={bias_sq:8.1f} "
        f"variance={variance:8.2f} "
        f"expected_RMSE={(bias_sq + variance) ** 0.5:6.2f} "
        f"({time.time() - t0:.0f}s)"
    )

decomp_df = pd.DataFrame(decomp)
display(decomp_df)

LinearRegression       bias^2(+noise)=  4652.5 variance=   36.07 expected_RMSE= 68.47 (2s)
Ridge                  bias^2(+noise)=  4642.7 variance=   39.37 expected_RMSE= 68.43 (1s)
RandomForest           bias^2(+noise)=  7328.1 variance=  160.52 expected_RMSE= 86.54 (103s)
SVR(rbf)               bias^2(+noise)=  7146.4 variance=   37.23 expected_RMSE= 84.76 (473s)
MLPAdam                bias^2(+noise)=  3163.7 variance=  109.93 expected_RMSE= 57.22 (85s)
MLP-LBFGS              bias^2(+noise)=  3844.7 variance= 2040.55 expected_RMSE= 76.72 (746s)
HistGradientBoosting   bias^2(+noise)=  3441.1 variance=  176.79 expected_RMSE= 60.15 (20s)


,model,bias_sq_plus_noise,variance,expected_mse,expected_rmse,time_s
0,LinearRegression,4652.502574,36.069011,4688.571585,68.473145,2.222470
1,Ridge,4642.673704,39.365525,4682.039230,68.425428,1.146596
2,RandomForest,7328.105626,160.520650,7488.626276,86.536849,102.510047
3,SVR(rbf),7146.420699,37.229375,7183.650074,84.756416,472.890328
4,MLPAdam,3163.745659,109.926178,3273.671837,57.216010,85.240078
5,MLP-LBFGS,3844.667686,2040.552762,5885.220448,76.715190,746.139848
6,HistGradientBoosting,3441.138899,176.790160,3617.929060,60.149223,19.965713
